# MIT ADT UNIVERSITY, PUNE
## Department of Computer Science & Engineering
### Course: Unit IV -- Multi-Agent Systems and Collaboration
### Assignment No: 04
**Title:** Collaborative Multi-Agent Team for Complex Task Completion using AutoGen

**Mapped Course Outcome:** `CO4 -- Design collaborative multi-agent systems for complex tasks. (Create)`

---
### Objective:
To enable students to design a team of role-based agents that communicate, coordinate, and negotiate to jointly solve a complex task, demonstrating multi-agent communication protocols and coordination strategies.

### Problem Statement:
Using AutoGen, design a multi-agent system of at least 3 role-based agents (Researcher/Architect, Writer/Coder, Reviewer/Critic) that collaborate end-to-end to complete a complex task (Enterprise Distributed Rate Limiter & Circuit Breaker Engine in Python). The system must include agent-to-agent communication, a coordination/negotiation mechanism, at least one human-in-the-loop checkpoint, a conflict scenario resolution, and a 1-page comparison report against a single-agent baseline.

## 1. Environment Setup & Library Imports

In [ ]:
# Import necessary libraries and check AutoGen version
import os
import asyncio
import json
from pathlib import Path
import autogen_agentchat
import autogen_core
import config
import tools
import communication as comm
import agents
import coordination
import baseline

print(f'AutoGen AgentChat Version: {autogen_agentchat.__version__}')
print(f'AutoGen Core Version: {autogen_core.__version__}')
print(f'Active LLM Model: {config.DEFAULT_MODEL}')
print('All modules loaded successfully!')

## 2. Specialized Tools Definition
Each agent is equipped with distinct, domain-specific tools:
- `search_technical_standards`: RFC & architectural pattern retrieval (SystemArchitect)
- `execute_python_code`: Isolated sandbox test execution (CodeEngineer)
- `security_ast_scanner`: Static AST security & concurrency linting (SecurityCritic)

In [ ]:
# Demonstrate tool execution
print('--- 1. Testing Search Tool ---')
search_res = tools.search_technical_standards('rate_limiter')
print(search_res[:250] + '...')

print('\n--- 2. Testing Sandbox Execution Tool ---')
exec_res = tools.execute_python_code('assert 2 * 3 == 6\nprint("Sandbox test passed!")')
print(exec_res)

print('\n--- 3. Testing AST Security Scanner ---')
scan_res = tools.security_ast_scanner('import threading\nx = 1 # Missing lock')
print(scan_res)

## 3. Communication Protocol: Message Passing & Shared Task Board
Implements task #9:
- `AgentMessage`: Structured messages with sender, receiver, message_type, metadata, and timestamps.
- `MessageBus`: Central routing bus capturing complete verifiable logs.
- `SharedTaskBoard`: Blackboard pattern tracking lifecycle states (`BACKLOG -> IN_PROGRESS -> REVIEW -> HUMAN_APPROVAL -> DONE`).

In [ ]:
bus = comm.MessageBus()
board = comm.SharedTaskBoard()

# Register task
board.add_task('TASK-01', 'System Spec', 'Draft token bucket design', 'SystemArchitect')
board.update_task_status('TASK-01', comm.TaskStatus.IN_PROGRESS, 'SystemArchitect', 'Began RFC analysis')

# Send sample message
msg = bus.send(comm.AgentMessage(
    sender='TeamManager',
    receiver='SystemArchitect',
    message_type=comm.MessageType.TASK_ASSIGNMENT,
    content='Initiate architectural specification for rate limiter.'
))
print(f'Message Routed: [{msg.timestamp}] {msg.sender} -> {msg.receiver} ({msg.message_type.value})')
print('Task Board Summary:', board.get_board_summary())

## 4. Role-Based Agent Setup (4 Specialized Agents)
Implements task #8:
1. **SystemArchitect**: Researches patterns, formulates class contracts & concurrency invariants.
2. **CodeEngineer**: Implements thread-safe code and executes tests in the sandbox.
3. **SecurityCritic**: Performs AST scans, audits for race conditions (TOCTOU), and issues verdicts.
4. **TeamManager**: Coordinates workflow, arbitrates conflicts, and synthesizes final releases.

In [ ]:
architect = agents.create_system_architect_agent()
engineer = agents.create_code_engineer_agent()
critic = agents.create_security_critic_agent()
manager = agents.create_orchestrator_agent()

team = [architect, engineer, critic, manager]
for a in team:
    print(f'Agent: {a.name:<16} | Description: {a.description}')

## 5. End-to-End Multi-Agent Pipeline with Human-in-the-Loop Checkpoint
Implements tasks #10, #11, and #12:
- Executes the 5-stage collaborative workflow
- Human-in-the-loop checkpoint validates intermediate architecture before code implementation

In [ ]:
# Load and inspect the verified task run log generated by the multi-agent system
log_path = config.LOGS_DIR / 'task_run_log.json'
with open(log_path, 'r', encoding='utf-8') as f:
    run_data = json.load(f)

print('=== MULTI-AGENT EXECUTION SUMMARY ===')
print(f'Human Checkpoint Decision: {run_data["human_decision"]}')
print(f'Human Feedback: {run_data["human_feedback"]}')
print(f'Total Inter-Agent Messages: {run_data["messages_count"]}')

print('\n=== FINAL TASK BOARD STATE ===')
for tid, card in run_data['task_board'].items():
    print(f'[{tid}] {card["title"]} -> Status: {card["status"]} (Assigned: {card["assigned_to"]})')

### 5.1 Verified Implementation Code Produced by CodeEngineer

In [ ]:
print(run_data['implementation_code'][:2500] + '\n... [code continues with full unit tests]')


### 5.2 Security & QA Audit Report Produced by SecurityCritic

In [ ]:
print(run_data['security_audit'])

## 6. Conflict Scenario & Dialectic Arbitration
Implements task #13:
- **Conflict Scenario**: CodeEngineer proposes lockless in-memory cache for speed; SecurityCritic rejects it due to race condition.
- **Resolution**: TeamManager mediates a 3-step dialectic arbitration, arriving at a partitioned lock-striping consensus.

In [ ]:
conflict_log = config.LOGS_DIR / 'conflict_resolution_log.json'
with open(conflict_log, 'r', encoding='utf-8') as f:
    c_data = json.load(f)

print(f'Scenario: {c_data["conflict_description"]}')
print(f'Resolution Status: {c_data["status"]}\n')
print('--- 1. Thesis (CodeEngineer Proposal) ---')
print(c_data['engineer_proposal'])

print('--- 2. Antithesis (SecurityCritic Objection) ---')
print(c_data['critic_objection'])

print('--- 3. Synthesis (TeamManager Arbitration Verdict) ---')
print(c_data['arbitration_verdict'])

## 7. Comparative Benchmark: Multi-Agent Team vs. Single-Agent Baseline
Implements task #14:
- Compares collaborative system against monolithic baseline across 7 technical dimensions.
- Plots quantitative evaluation metrics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

chart_file = config.BASE_DIR / 'metrics_comparison_chart.png'
if chart_file.exists():
    img = mpimg.imread(str(chart_file))
    plt.figure(figsize=(12, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

eval_file = config.LOGS_DIR / 'comparison_evaluation.json'
with open(eval_file, 'r', encoding='utf-8') as f:
    ev = json.load(f)

print(f'Single-Agent Baseline Score: {ev["summary"]["single_agent_overall"]} / 10')
print(f'Multi-Agent Team Score:     {ev["summary"]["multi_agent_overall"]} / 10')
print(f'Quality Improvement:        +{ev["summary"]["quality_improvement_percent"]} %')

## 8. Conclusion & Mapped Course Outcome (CO4) Reflection

1. **Course Outcome CO4 Achieved**: Successfully designed, implemented, and evaluated a collaborative multi-agent system.
2. **Effective Division of Labor**: Decoupling architecture, coding, and auditing prevented cognitive hallucination and produced production-grade code.
3. **Human-in-the-Loop Safety**: Intermediate approval checkpoints guarantee alignment and prevent unconstrained execution.
4. **Dialectic Convergence**: Demonstrated that multi-agent systems can autonomously detect and reconcile conflicting technical requirements through structured arbitration.